# Exercise 2 - Task-based laminar fMRI analysis

Welcome to the second exercise of the laminar fMRI course.

In this notebook you will be looking at a dataset from a published paper: https://www.nature.com/articles/s42003-024-06780-8
In this experiment, participants completed a working memory task where they had to maintain either many or a single item in mind for 12s each trial. 
We will take a look at how the laminar activity in the dorso-lateral prefrontal cortex (dlPFC), a region known to be engaged in working memory processes, changes depending on whether many (high load) or a single (low load) item is maintained. 

More specifically, we will:

- extract **High** and **Low** load responses from a chosen ROI. For those interested, these are generated using AFNI's 3dDeconvolve (where FIR models are fit to measure the time course)
- average the signal within **three cortical layers** within this ROI
- plot **group time courses** averaged across subjects
- visualise the **load effect** (*High - Low*) with **violin plots**
- run a **paired t-test** on the layer-specific load difference during part of the delay period

In the end, you should be able to reproduce one of the results from the original paper. 


## Data layout used in this exercise

Each participant lives in a subject folder such as:

```text
/data_ex2/S01/ through /data_ex2/S09/
```

Relevant files in each subject's folder:

- `/func/Load_long_TENTzero_response_condition_High_prcchg.nii`
- `/func/Load_long_TENTzero_response_condition_Low_prcchg.nii`

ROI masks in `/anat/` of each subject's folder:

- `dlpfc_l_parcel_map.nii` -> left DLPFC
- `cop_l_parcel_map.nii` -> left COP
- `cop_r_parcel_map.nii` -> right COP
- `fpn_r_parcel_map.nii` -> right DLPFC

Layer map in `/anat/`:

- `ds_scaled_rim_layers_equidist_3layers.nii`

The layer labels are:

- **1 = Deep**
- **2 = Middle**
- **3 = Superficial**

## Analysis strategy

For each subject and ROI:

1. load the ROI mask
2. load the 3-layer layer file
3. intersect the ROI with the layer file
4. average the functional response within each layer
5. plot the time course for high and low load for each layer

At the group level we then:

- compute the mean time course across subjects
- show the **SEM** around the mean
- compute the load effect as **High - Low** during the delay period (specific selected time points)

In [3]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from laminar_ex2_utils import (
    DEFAULT_PARTICIPANTS,
    load_group_data,
    plot_group_timecourses,
    plot_load_effect_violin,
    paired_ttest_between_layers,
    print_ttest_report,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 1. Set the data location and choose an ROI

By default we start with **left DLPFC**.

- `"dlpfc_left"`

You can also choose one of these alternatives:

- `"cop_left"`
- `"cop_right"`
- `"dlpfc_right"`

The dlPFC mask is defined as parcels belonging to the fronto-parietal network, while the COP stands for the cingulo-opercular network which also consists of frontal cortex parcels. 
Right/left indicates the hemisphere of the cortex.

In [ ]:
data_dir = Path("../layer-course-data/data_ex2")
participants = DEFAULT_PARTICIPANTS

roi = "dlpfc_left"   # default ROI for this exercise
group_data = load_group_data(data_dir=data_dir, participants=participants, roi=roi)

print("Loaded ROI:", group_data["roi"])
print("Subjects:", ", ".join(group_data["participants"]))
print("High shape  (subjects x layers x time):", group_data["high"].shape)
print("Low shape   (subjects x layers x time):", group_data["low"].shape)
print("Time axis (s):", group_data["time_seconds"])


## 2. Plot group time courses

The plotting function:

- allows you to select **which layers** to display
- places layers in **horizontal subplots**
- uses one **global amplitude range** across the displayed layers
- adds **SEM shading** around the mean
- (specific to the AFNI analysis: keeps the **TENTzero** zeros at the beginning and end)

Because each time point corresponds to **2 s**, the x-axis is already shown in seconds.


In [ ]:
# Plot all layers
fig, axes = plot_group_timecourses(
    group_data,
    layers_to_plot=(1, 2, 3),   # choose any subset, e.g. (1, 3)
    conditions=("high", "low"),
    figsize=(15, 4.5),
)
plt.show()

### Example: plot only selected layers

This is useful when you want to focus on a subset of the cortical depth profile.

In [ ]:
fig, axes = plot_group_timecourses(
    group_data,
    layers_to_plot=(1, 3),   # deep and superficial only
    conditions=("high", "low"),
    figsize=(10, 4.5),
)
plt.show()

## 3. Plot the load effect

Next we summarise part of the delay period.

Here we use **points 4 to 6** and that correspond to the start of the delay period (when memory is maintained). We compute the load effect during this time:

```python
High - Low
```

for each subject and each layer.

The plot shows:

- one violin per layer
- transparent lines connecting each subject across layers

In [ ]:
delay_window = (5, 7)

fig, ax = plot_load_effect_violin(
    group_data,
    time_window=delay_window,
    layers=(1, 3),
    figsize=(8, 5),
)
plt.show()


## 4. Run a paired t-test between two layers

Here the statistical question is:

> Is the **load difference** (*High - Low*) different between **deep* and **superficial layers* in a the left dlPFC** significant?

We use a **within-subject paired t-test**.

In [ ]:
ttest_result = paired_ttest_between_layers(
    group_data,
    layer_x=1,          # Deep
    layer_y=3,          # Superficial
    time_window=(5, 7),
)

print_ttest_report(ttest_result)


## 5. Try a different ROI

To switch ROI, simply reload the group data with another ROI label.


In [ ]:
roi = "cop_left"   # try: "cop_left", "cop_right", "dlpfc_right"
group_data = load_group_data(data_dir=data_dir, participants=participants, roi=roi)

fig, axes = plot_group_timecourses(group_data, layers_to_plot=(1, 2, 3))
plt.show()

fig, ax = plot_load_effect_violin(group_data, time_window=(4, 6))
plt.show()

ttest_result = paired_ttest_between_layers(group_data, layer_x=1, layer_y=3, time_window=(4, 6))
print_ttest_report(ttest_result)


## Notes for interpretation

A few practical reminders for students:

- The time courses are **ROI averages**, not voxel-wise analyses.
- If the t-test is significant, that suggests the load modulation differs between the two chosen layers in that ROI and time window.